# NHS A&E Performance Analysis
**Author:** Husnain Zahoor  
**Date:** Aug 05, 2026    
**Dataset:** NHS A&E Quality Indicators (Provisional, December 2023 – December 2025)  
**Source URL:** https://tinyurl.com/NHS-Source-Data

---

### Context & Pipeline Position
This notebook (2 of 4) loads `df_final.pkl` from Notebook 01 to construct derived performance metrics, operational categories, and regional aggregations. It creates the enriched analysis dataset (`df_analysis.pkl`) and generates key statistical summaries for stakeholder reporting.

###**Executive Summary: Operational Analysis of NHS A&E Quality Indicators**

This analysis looks at how NHS emergency departments are performing, and
whether treatment delays are linked to patients returning within 7 days.
It covers 140 NHS Trusts over 25 months, from December 2023 to December
2025, focusing on wait times, assessment delays, treatment speed, and
reattendance rates.

###**Key analytical findings** :
Three findings stand out:

1. Blackpool Teaching Hospitals had the longest average wait at 352 minutes, more than double the national median of 169 minutes, and 83 minutes higher than the next-worst Trust.


2. National wait times improved from 186 minutes in December 2023
to around 160 minutes by mid-2024, and have stayed at that level since.


3. Faster treatment does not guarantee fewer return visits, some
Trusts with quick treatment times still have high reattendance rates.

###**Recommendation:**
Review Blackpool's emergency department directly, and
check discharge processes at Trusts with reattendance rates above 10%.

## Environment Setup
Mount Google Drive and define standard file paths. Run these three cells at the start of every session.

In [ ]:
# Mount Google Drive — run this first in every session
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define standard paths — reference these throughout the module
DATA_PATH   = '/content/drive/My Drive/Lumen/python-data-analytics/Data/'
OUTPUT_PATH = '/content/drive/My Drive/Lumen/python-data-analytics/Output/'

In [ ]:
# Verify setup — confirm all three CSV files are accessible
import os
print(os.listdir(DATA_PATH))

['aeqi_metadata.csv', 'aeqi_open_data_2025_12.csv', 'nhs_trust_reference.csv']


## Library Imports

In [ ]:
# Setup and Dependencies
import pandas as pd
import numpy as np

# FORCE PANDAS TO SHOW ALL COLUMNS HORIZONTALLY (NO WRAPPING)
pd.set_option('display.max_columns', None)  # Show every column
pd.set_option('display.width', 1000)        # Give the text engine plenty of horizontal width

In [ ]:
# We are working from df_final but naming it df_analysis to signal that this is the enriched analytical version.
# df_final remains unchanged as your clean base.

df_analysis = pd.read_pickle(OUTPUT_PATH + 'df_final.pkl')
print(df_analysis.shape)
print(df_analysis.dtypes)

(78386, 16)
ATTENDANCE_MONTH    datetime64[ns]
ORG_CODE                    object
ORG_NAME                    object
MEASURE_ID                  object
MEASURE_NAME                object
MEASURE_VALUE              float64
SUPPRESSION                 object
DESCRIPTION                 object
DATA_TYPE                   object
CAVEATS                     object
SPECIFICATION               object
NHSER_CODE                  object
REGION                      object
ICB_CODE                    object
OPEN_DATE                  float64
CLOSE_DATE                 float64
dtype: object


## Create a breach_flag column


In [ ]:
# Review what DATA_TYPE values exist
print(df_analysis['DATA_TYPE'].value_counts())

# Review MEASURE_IDs that relate to time
time_measures_raw = df_analysis[df_analysis['MEASURE_NAME'].str.contains('TIME|MEDIAN|MINUTES', case=False, na=False)]['MEASURE_ID'].unique()
print(f"Time realted measures: {time_measures_raw}")

DATA_TYPE
Decimal    44238
Integer    34148
Name: count, dtype: int64
Time realted measures: ['AEQI053' 'AEQI054' 'AEQI058' 'AEQI055' 'AEQI056' 'AEQI059' 'AEQI051'
 'AEQI052' 'AEQI057' 'AEQI041' 'AEQI042' 'AEQI043' 'AEQI031' 'AEQI032'
 'AEQI033']


In [ ]:
# Exclude denominators — these are patient counts, not minutes
# Including them would inflate breach_flag against a 240-minute threshold
denominator_ids = ['AEQI033', 'AEQI043', 'AEQI057', 'AEQI058', 'AEQI059']

time_measures = np.setdiff1d(time_measures_raw, denominator_ids)

print("Final time measures:", time_measures)
print("Count:", len(time_measures))

Final time measures: ['AEQI031' 'AEQI032' 'AEQI041' 'AEQI042' 'AEQI051' 'AEQI052' 'AEQI053'
 'AEQI054' 'AEQI055' 'AEQI056']
Count: 10


In [ ]:
# np.where(condition, value_if_true, value_if_false)
df_analysis['is_time_measure'] = np.where(df_analysis['MEASURE_ID'].isin(time_measures),True,False)

print(df_analysis['is_time_measure'].value_counts())

is_time_measure
False    44488
True     33898
Name: count, dtype: int64


In [ ]:
# breach_flag: 1 if time measure AND value exceeds 240 minutes, else 0
df_analysis['breach_flag'] = np.where((df_analysis['is_time_measure'] == True) & (df_analysis['MEASURE_VALUE'] > 240),1,0)

n_breaches = df_analysis['breach_flag'].sum()
pct_breaches = n_breaches / df_analysis[df_analysis['is_time_measure']].shape[0] * 100
print(f"Total breach rows: {n_breaches}")
print(f"As % of time-measure rows: {pct_breaches:.1f}%")

Total breach rows: 14210
As % of time-measure rows: 41.9%


The 240-minute threshold represents the NHS four-hour A&E standard: 95% of patients should be seen, treated, and admitted or discharged within 240 minutes. A Trust-month value above this threshold is a breach of the national performance standard.

Applying this flag to non-time measures would be wrong because MEASURE_VALUE holds all 23 KPIs in one column (long format). A reattendance rate of 300 (AEQI061) or a denominator count of 5,000 (AEQI043) would get flagged as a breach, even though neither has anything to do with waiting time.

String matching on MEASURE_NAME using 'TIME|MEDIAN|MINUTES' also picked up denominator KPIs (AEQI033, AEQI043, AEQI057, AEQI058, AEQI059) — their names mention time, but their values are patient counts, not minutes. These were excluded after checking the data dictionary. Without this fix, the breach rate came out at 60.3% instead of the correct 41.9% — an 18-point overstatement that would give the Head of A&E Operations a wrong picture of performance.

##Create a performance_band column using .loc[]

In [ ]:
time_mask = df_analysis['is_time_measure'] == True

df_analysis.loc[time_mask & (df_analysis['MEASURE_VALUE'] <= 200), 'performance_band'] = 'Good'
df_analysis.loc[time_mask & (df_analysis['MEASURE_VALUE'] > 200) & (df_analysis['MEASURE_VALUE'] <= 240),'performance_band'] = 'Acceptable'
df_analysis.loc[time_mask & (df_analysis['MEASURE_VALUE'] > 240), 'performance_band'] = 'Poor'

print(df_analysis['performance_band'].value_counts(dropna=False))

performance_band
NaN           44488
Good          16933
Poor          14210
Acceptable     2755
Name: count, dtype: int64


**Performance band classification:**

Used .loc[] rather than a custom function because the logic is simple: fixed numeric thresholds, small number of categories. A user-defined function would add unnecessary complexity for a three-line rule.

Distribution across performance bands:

Poor: 42% of time-measure rows — most NHS Trusts, across most months in this dataset, are breaching the 4-hour standard.
Acceptable: 8% — very few Trusts sit in the middle. Trusts are either meeting the target comfortably or missing it badly.
Good: 16,933 rows (50%) — half the dataset meets the standard.

The small Acceptable band is the key signal: performance is polarised, not gradual. Trusts don't drift near the target — they're either clearly on track or clearly failing it.

##Create a month_name column

In [ ]:
# Extract month name and year from ATTENDANCE_MONTH (datetime column)
df_analysis['month_name'] = df_analysis['ATTENDANCE_MONTH'].dt.strftime('%B %Y')

# Verify the unique values and sort them chronologically
print(df_analysis.sort_values('ATTENDANCE_MONTH')['month_name'].unique())

['December 2023' 'January 2024' 'February 2024' 'March 2024' 'April 2024'
 'May 2024' 'June 2024' 'July 2024' 'August 2024' 'September 2024'
 'October 2024' 'November 2024' 'December 2024' 'January 2025'
 'February 2025' 'March 2025' 'April 2025' 'May 2025' 'June 2025'
 'July 2025' 'August 2025' 'September 2025' 'October 2025' 'November 2025'
 'December 2025']


The raw datetime column displays as 2024-12-01 00:00:00 — technically
correct, but not readable for a stakeholder chart axis. month_name
formats the same value as December 2024, which the Head of A&E
Operations can read instantly.

Sorting happens on ATTENDANCE_MONTH (datetime) before month_name is
generated. This keeps chronological order. If month_name were sorted
directly, it would sort alphabetically — April before December,
regardless of year.

This column labels the x-axis in the attendance trend charts, so
winter peaks like December 2024 and January 2025 are readable at a
glance, without decoding timestamp formats.

##Create an is_winter column

In [ ]:
# Define winter months
winter_months = [10, 11, 12, 1, 2, 3]

# Extract numeric month and check if it falls in the winter list
df_analysis['is_winter'] = df_analysis['ATTENDANCE_MONTH'].dt.month.isin(winter_months)

print(df_analysis['is_winter'].value_counts())
print()
# Cross-check: which months are flagged as winter?
print(df_analysis.groupby('is_winter')['month_name'].unique())

is_winter
True     40743
False    37643
Name: count, dtype: int64

is_winter
False    [August 2025, July 2024, August 2024, Septembe...
True     [February 2024, March 2024, February 2025, Jan...
Name: month_name, dtype: object


NHS England starts winter planning in October because flu season,
respiratory illness, and cold weather frailty presentations begin
rising before December. Waiting until December to activate winter
protocols would mean departments are already overwhelmed before any
response is mobilised.

The is_winter flag marks 40,743 Trust-month-KPI rows across
October–March. Higher values are expected in AEQI051 (median total
time), higher rates in breach_flag, and a shift in performance_band
toward "Poor" — reflecting the pressure that fixed staffing and bed
capacity face against seasonal demand. This flag allows a direct
winter vs summer comparison in the grouping and aggregation stage
without extra filtering logic.

##Write a user-defined function for reattendance classification

In [ ]:
# Define the function first:
def classify_reattendance(rate):

    if pd.isna(rate):
        return np.nan
    elif rate < 8:
        return 'Low'
    elif rate <= 12:
        return 'Medium'
    else:
        return 'High'

In [ ]:
# Apply only to rows where MEASURE_ID is AEQI062
mask_aeqi062 = df_analysis['MEASURE_ID'] == 'AEQI062'

df_analysis.loc[mask_aeqi062, 'reattendance_band'] = (df_analysis.loc[mask_aeqi062, 'MEASURE_VALUE'].apply(classify_reattendance))

print(df_analysis.loc[mask_aeqi062, 'reattendance_band'].value_counts())

print()
print()

print(f"Rows classified: {mask_aeqi062.sum()}")
print(f"Rows not applicable (other measures): {(~mask_aeqi062).sum()}")

reattendance_band
Medium    2043
Low       1128
High       207
Name: count, dtype: int64


Rows classified: 3378
Rows not applicable (other measures): 75008


##Regional performance summary.

In [ ]:
aeqi051 = df_analysis[df_analysis['MEASURE_ID'] == 'AEQI051']

regional_summary = aeqi051.groupby('REGION').agg({'MEASURE_VALUE': ['mean','median','std']})

regional_summary.columns = ['mean_region','median_region','std_region']
regional_summary = regional_summary.reset_index()
print(regional_summary.sort_values('mean_region', ascending=False))

                     REGION  mean_region  median_region  std_region
0           East of England   186.801538          186.0   30.953991
4                North West   185.610586          183.0   64.504856
1                    London   183.448000          180.0   70.910436
2                  Midlands   176.389167          184.0   47.114207
6                South West   170.285333          183.0   49.131439
5                South East   163.851818          176.5   52.915115
3  North East and Yorkshire   163.292174          168.0   49.941785


East of England has the worst mean performance across regions. London's
result is unreliable at the regional level — its standard deviation is
too high, so the average hides big differences between Trusts. London
needs Trust-level investigation, not a regional average, before drawing
conclusions.

##Trust-level reattendance outliers.

In [ ]:
aeqi062  = df_analysis[df_analysis['MEASURE_ID'] == 'AEQI062']
trust_level_reattendance  = aeqi062.groupby(['ORG_CODE','ORG_NAME']).agg({'MEASURE_VALUE': ['mean','median','max','std']})
trust_level_reattendance.columns = ['mean_reattandance','median_reattandance','max_reattandance','std_reattandance']
trust_level_reattendance  = trust_level_reattendance .reset_index()
print(f"Top 10 Rows:")
print(trust_level_reattendance.sort_values('mean_reattandance', ascending=False).head(10))
print(f"\nBottom 10 Rows:")
print(trust_level_reattendance.sort_values('mean_reattandance', ascending=True).head(10))

Top 10 Rows:
    ORG_CODE                                           ORG_NAME  mean_reattandance  median_reattandance  max_reattandance  std_reattandance
93       RTF        NORTHUMBRIA HEALTHCARE NHS FOUNDATION TRUST          15.153425            15.023232         16.460474          0.751356
111      RWF            MAIDSTONE AND TUNBRIDGE WELLS NHS TRUST          14.674176            14.597060         15.612700          0.365232
36       RF4  BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...          13.610032            13.739968         15.520221          1.027864
107      RW4                   MERSEY CARE NHS FOUNDATION TRUST          12.577785            12.689225         14.827586          1.327176
35       REP             LIVERPOOL WOMEN'S NHS FOUNDATION TRUST          12.426457            12.500000         14.234875          1.220738
122      RXK   SANDWELL AND WEST BIRMINGHAM HOSPITALS NHS TRUST          12.378081            12.099359         24.972406          2.960073
127    

A high standard deviation means a Trust's monthly reattendance rate
swings widely instead of staying steady.

Sandwell and West Birmingham has a CV of 23.9% (2.96 / 12.38), which
is moderate-high — meaning the rate is not fully consistent. But the
max value of 24.97% sits 4.25 standard deviations above the mean
((24.97 - 12.38) / 2.96 = 4.25). A gap this large points to one
severe spike month, not a Trust that is unstable every month. This
needs a month-by-month check to find out what happened in that spike
month.

##Trust–KPI pivot table.

In [ ]:
# Pivot: rows = ORG_CODE, columns = MEASURE_ID, values = mean MEASURE_VALUE
pivot_trust = df_analysis.pivot_table(index='ORG_CODE',columns='MEASURE_ID',values='MEASURE_VALUE',aggfunc='mean').reset_index()

print(pivot_trust.shape)
pivot_trust.head()

(140, 24)


MEASURE_ID,ORG_CODE,AEQI011,AEQI012,AEQI013,AEQI021,AEQI022,AEQI031,AEQI032,AEQI033,AEQI041,AEQI042,AEQI043,AEQI051,AEQI052,AEQI053,AEQI054,AEQI055,AEQI056,AEQI057,AEQI058,AEQI059,AEQI061,AEQI062,AEQI063
0,R0A,43638.0,44028.12,99.108867,1947.0,4.455901,5.52000,37.858000,6261.8,73.76,324.464,26699.8,182.76,725.802,299.22,1039.966,161.60,496.400,43620.4,11142.4,32478.4,3440.2,9.939107,34634.6
1,R0B,21717.8,20603.48,105.426420,994.4,4.582026,7.16000,48.620000,4259.0,79.68,321.188,17848.4,161.16,576.044,277.18,767.978,137.68,447.882,21705.6,5936.4,15769.2,2046.0,10.450682,19571.2
2,R0D,16234.4,18402.48,88.212324,927.2,5.681098,1.00000,4.480000,4185.6,77.36,377.400,12510.4,192.94,776.636,400.52,1088.794,149.92,540.766,16233.6,4009.0,12224.6,1113.0,8.042681,13802.2
3,R1D,3132.8,3120.88,100.390880,43.8,1.369785,55.71875,74.646875,0.0,0.00,0.000,3088.6,85.80,224.702,73.56,213.708,87.00,225.496,3132.8,271.8,2860.8,109.0,4.180421,2612.6
4,R1F,5020.6,6109.84,82.060766,137.2,2.740192,7.96000,17.040000,1474.6,60.72,198.608,4517.6,210.28,1216.780,574.90,1671.476,182.54,580.976,5020.6,1075.6,3944.8,419.6,9.127544,4592.6


In [ ]:
# Bring in org name and region for readability
org_lookup = df_analysis[['ORG_CODE', 'ORG_NAME', 'REGION']].drop_duplicates()

pivot_trust = pivot_trust.merge(org_lookup, on='ORG_CODE', how='left')

# Reorder columns: identifiers first
id_cols = ['ORG_CODE', 'ORG_NAME', 'REGION']
kpi_cols = [c for c in pivot_trust.columns if c not in id_cols]

pivot_trust = pivot_trust[id_cols + kpi_cols]

print(pivot_trust[['ORG_NAME', 'REGION', 'AEQI051', 'AEQI062']].head(10))

pivot_trust.to_csv(OUTPUT_PATH + 'pivot_trust_kpi.csv', index=False)
print(f"\nPivot table saved successfully:")
print(pivot_trust.shape)

                                            ORG_NAME                    REGION  AEQI051    AEQI062
0         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West   182.76   9.939107
1  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire   161.16  10.450682
2   UNIVERSITY HOSPITALS DORSET NHS FOUNDATION TRUST                South West   192.94   8.042681
3              SHROPSHIRE COMMUNITY HEALTH NHS TRUST                  Midlands    85.80   4.180421
4                            ISLE OF WIGHT NHS TRUST                South East   210.28   9.127544
5                             BARTS HEALTH NHS TRUST                    London   234.12  10.107377
6  LONDON NORTH WEST UNIVERSITY HEALTHCARE NHS TRUST                    London   170.68  10.438947
7                  ROYAL SURREY NHS FOUNDATION TRUST                South East   181.26   2.836675
8  UNIVERSITY HOSPITALS BRISTOL AND WESTON NHS FO...                South West   194.24  10.304613
9        T

##Regional benchmarking with transform().

In [ ]:
aeqi051 = df_analysis[df_analysis['MEASURE_ID'] == 'AEQI051'].copy()

aeqi051['regional_mean_wait'] = aeqi051.groupby('REGION')['MEASURE_VALUE'].transform('mean')
print(aeqi051[['ORG_NAME','REGION','MEASURE_VALUE','regional_mean_wait']].head(15))

                                                ORG_NAME                    REGION  MEASURE_VALUE  regional_mean_wait
24939         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          196.0          185.610586
24940         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          187.0          185.610586
24941         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          183.0          185.610586
24942  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          171.0          163.292174
24943  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          173.0          163.292174
24944  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          151.0          163.292174
24945  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          164.0          163.292174
24946  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...

In [ ]:
aeqi051['vs_region'] = aeqi051['MEASURE_VALUE'] - aeqi051['regional_mean_wait']
print(aeqi051[['ORG_NAME','REGION','MEASURE_VALUE','regional_mean_wait', 'vs_region']].head(15))

                                                ORG_NAME                    REGION  MEASURE_VALUE  regional_mean_wait  vs_region
24939         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          196.0          185.610586  10.389414
24940         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          187.0          185.610586   1.389414
24941         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          183.0          185.610586  -2.610586
24942  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          171.0          163.292174   7.707826
24943  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          173.0          163.292174   9.707826
24944  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          151.0          163.292174 -12.292174
24945  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          164.0

In [ ]:
aeqi051['regional_band'] = 'Average'
aeqi051.loc[aeqi051['MEASURE_VALUE'] < aeqi051['regional_mean_wait'] * 0.9, 'regional_band'] = 'Better than region'
aeqi051.loc[aeqi051['MEASURE_VALUE'] > aeqi051['regional_mean_wait'] * 1.1, 'regional_band'] = 'Worse than region'

print(aeqi051[['ORG_NAME','REGION','MEASURE_VALUE','regional_mean_wait','regional_band']].head(15))

                                                ORG_NAME                    REGION  MEASURE_VALUE  regional_mean_wait       regional_band
24939         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          196.0          185.610586             Average
24940         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          187.0          185.610586             Average
24941         MANCHESTER UNIVERSITY NHS FOUNDATION TRUST                North West          183.0          185.610586             Average
24942  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          171.0          163.292174             Average
24943  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          173.0          163.292174             Average
24944  SOUTH TYNESIDE AND SUNDERLAND NHS FOUNDATION T...  North East and Yorkshire          151.0          163.292174             Average
24945  SOUTH TYNESIDE AND SUNDERLA

Threshold used: ±10% from the regional mean. A Trust more than 10%
above its regional mean is flagged 'Worse than region'. This range
is wide enough to avoid flagging normal month-to-month noise, but
tight enough to catch real performance gaps. This is a working
assumption — it would need sign-off from the clinical lead before
this threshold goes into a published report.

##National and regional rankings.

In [ ]:
# Always rebuild from df_final, never merge into an existing pivot_trust
pivot_trust = df_analysis.pivot_table(index='ORG_CODE', columns='MEASURE_ID', values='MEASURE_VALUE', aggfunc='mean').reset_index()

org_lookup = df_analysis[['ORG_CODE', 'ORG_NAME', 'REGION']].drop_duplicates()
pivot_trust = pivot_trust.merge(org_lookup, on='ORG_CODE', how='left')

print(pivot_trust.shape)  # should be (140, 26) every time you run this

pivot_trust['national_rank'] = pivot_trust['AEQI051'].rank(ascending=True, method='min')
pivot_trust['regional_rank'] = pivot_trust.groupby('REGION')['AEQI051'].rank(ascending=True, method='min')

top10 = pivot_trust.nsmallest(10, 'AEQI051')[['ORG_NAME', 'REGION', 'AEQI051', 'national_rank', 'regional_rank']]
print('Top 10 performers (shortest A&E wait):')
print(top10.to_string(index=False))

bottom10 = pivot_trust.nlargest(10, 'AEQI051')[['ORG_NAME', 'REGION', 'AEQI051', 'national_rank', 'regional_rank']]
print('Bottom 10 performers (longest A&E wait):')
print(bottom10.to_string(index=False))


pivot_trust.to_csv(OUTPUT_PATH + 'pivot_trust_ranked.csv', index=False)
print('pivot_trust_ranked.csv saved.')

(140, 26)
Top 10 performers (shortest A&E wait):
                                                 ORG_NAME                   REGION   AEQI051  national_rank  regional_rank
             QUEEN VICTORIA HOSPITAL NHS FOUNDATION TRUST               South East 44.125000            1.0            1.0
                     HUMBER TEACHING NHS FOUNDATION TRUST North East and Yorkshire 53.280000            2.0            1.0
               KENT COMMUNITY HEALTH NHS FOUNDATION TRUST               South East 61.660000            3.0            2.0
     GLOUCESTERSHIRE HEALTH AND CARE NHS FOUNDATION TRUST               South West 63.120000            4.0            1.0
              NORTHUMBRIA HEALTHCARE NHS FOUNDATION TRUST North East and Yorkshire 73.880000            5.0            2.0
DERBYSHIRE COMMUNITY HEALTH SERVICES NHS FOUNDATION TRUST                 Midlands 77.440000            6.0            1.0
            CENTRAL LONDON COMMUNITY HEALTHCARE NHS TRUST                   London 77.4800

**North West** accounts for **4** of the 10 worst-performing Trusts nationally, more than any other region, while Midlands (3), London (2), and North East and Yorkshire (1) make up the remainder.

This **concentration** is not visible in the regional averages from **Exercise 1**, where North West ranked only mid-table confirming that Trust-level outliers can be hidden inside a region's overall mean.

In [ ]:
df_analysis.to_pickle(OUTPUT_PATH + 'df_analysis.pkl')
print(f"Saved: df_analysis.pkl")
print(f"df_analysis.Shape: {df_analysis.shape}")
print(f"Columns: {df_analysis.columns.tolist()}")

Saved: df_analysis.pkl
df_analysis.Shape: (78386, 22)
Columns: ['ATTENDANCE_MONTH', 'ORG_CODE', 'ORG_NAME', 'MEASURE_ID', 'MEASURE_NAME', 'MEASURE_VALUE', 'SUPPRESSION', 'DESCRIPTION', 'DATA_TYPE', 'CAVEATS', 'SPECIFICATION', 'NHSER_CODE', 'REGION', 'ICB_CODE', 'OPEN_DATE', 'CLOSE_DATE', 'is_time_measure', 'breach_flag', 'performance_band', 'month_name', 'is_winter', 'reattendance_band']


In [ ]:
# Total Region
pivot_trust.groupby('REGION')['AEQI051'].count()

,AEQI051
REGION,
East of England,13
London,20
Midlands,24
North East and Yorkshire,23
North West,22
South East,23
South West,15


#**END**